In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pulp as plp

In [ ]:
capacity = 50.0  # kWh
initial_state_of_charge = 20.0  # kWh

forecast_horizon = 30
price_forecast = np.ones(shape=(forecast_horizon,)) * 5.0  # £ / kWh
demand_forecast = np.ones(shape=(forecast_horizon,)) * 10.0  # kWh 

In [ ]:
plt.plot(price_forecast, label="Price (£ / kWh)")
plt.plot(demand_forecast, label="Demand (kWh)")
plt.legend()

In [ ]:
# Greedy solver

def greedy(
    state_of_charge: float,
    demand: float,
    price: float,
    capacity: float = capacity
) -> float:

    problem = plp.LpProblem(name="battery_charging", sense=plp.LpMinimize)

    # Decision variable
    grid_flow = plp.LpVariable(name="grid_flow", lowBound=0, cat="Continuous")

    # Objective function
    problem += grid_flow * price

    # Constraints
    problem += grid_flow >= demand - state_of_charge
    problem += grid_flow <= capacity - state_of_charge + demand

    problem.solve(plp.PULP_CBC_CMD(msg=False))
    return grid_flow.varValue

In [ ]:
current_demand = 15
current_state_of_charge = 10
current_price = 10
print(f"Optimal grid inflow: {greedy(current_state_of_charge, current_demand, current_price)}")

In [ ]:
def transition(
    current_state_of_charge: float,
    grid_flow: float,
    demand_forecast: np.ndarray,
    price_forecast: np.ndarray,
):
    current_demand = demand_forecast[0]
    new_demand_forecast = demand_forecast[1:]
    current_price = price_forecast[0]
    new_price_forecast = price_forecast[1:]

    # Update state given current state and action
    new_state_of_charge = current_state_of_charge - current_demand + grid_flow

    # Compute total cost of charge
    cost_of_charge = grid_flow * current_price

    return (
        new_state_of_charge,
        new_demand_forecast,
        new_price_forecast,
        cost_of_charge
    )

In [ ]:
def simulate_greedy(
    demand_forecast: np.ndarray,
    price_forecast: np.ndarray,
    n_timesteps: int,
    initial_state_of_charge: float = initial_state_of_charge,
    capacity: float = capacity,
):
    
    grid_flow_history = []
    state_of_charge_history = [initial_state_of_charge]
    cost_history = []

    current_state_of_charge = initial_state_of_charge
    
    for t in range(n_timesteps):
        # Take action given current state
        grid_flow = greedy(
            state_of_charge=current_state_of_charge,
            demand=demand_forecast[0],
            price=price_forecast[0],
            capacity=capacity
        )
        grid_flow_history.append(grid_flow)

        # Transition to new state given action
        (
            current_state_of_charge,
            demand_forecast,
            price_forecast,
            cost_of_charge,
        ) = transition(
            current_state_of_charge=current_state_of_charge,
            grid_flow=grid_flow,
            demand_forecast=demand_forecast,
            price_forecast=price_forecast
        )
        state_of_charge_history.append(current_state_of_charge)
        cost_history.append(cost_of_charge)

    return (
        grid_flow_history,
        state_of_charge_history,
        cost_history,
    )

In [ ]:
(
    grid_flow_history,
    state_of_charge_history,
    cost_history,
) = simulate_greedy(
    demand_forecast=demand_forecast,
    price_forecast=price_forecast,
    n_timesteps=20,
)